In [55]:
import pandas as pd

In [56]:
df = pd.read_csv("../data/raw/male_players.csv")
df.head()

/var/folders/v6/c29cbybs71971d_cj04qx27m0000gn/T/ipykernel_81308/4078426669.py:1: DtypeWarning: Columns (108) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/male_players.csv")


,player_id,player_url,fifa_version,fifa_update,update_as_of,short_name,long_name,player_positions,overall,potential,...,ldm,cdm,rdm,rwb,lb,lcb,cb,rcb,rb,gk
0,231747,/player/231747/kylian-mbappe/240002,24.0,2.0,2023-09-22,K. Mbappé,Kylian Mbappé Lottin,"ST, LW",91,94,...,63+3,63+3,63+3,68+3,63+3,54+3,54+3,54+3,63+3,18+3
1,239085,/player/239085/erling-haaland/240002,24.0,2.0,2023-09-22,E. Haaland,Erling Braut Haaland,ST,91,94,...,63+3,63+3,63+3,62+3,60+3,62+3,62+3,62+3,60+3,19+3
2,192985,/player/192985/kevin-de-bruyne/240002,24.0,2.0,2023-09-22,K. De Bruyne,Kevin De Bruyne,"CM, CAM",91,91,...,80+3,80+3,80+3,79+3,75+3,70+3,70+3,70+3,75+3,21+3
3,158023,/player/158023/lionel-messi/240002,24.0,2.0,2023-09-22,L. Messi,Lionel Andrés Messi Cuccittini,"CF, CAM",90,90,...,63+3,63+3,63+3,64+3,59+3,49+3,49+3,49+3,59+3,19+3
4,165153,/player/165153/karim-benzema/240002,24.0,2.0,2023-09-22,K. Benzema,Karim Benzema,"CF, ST",90,90,...,64+3,64+3,64+3,64+3,60+3,55+3,55+3,55+3,60+3,18+3


In [57]:
df = df[df['fifa_version'] == df['fifa_version'].max()]

In [58]:
query_features = [
    "player_positions",
    "overall",
    "potential",
    "age",
    "value_eur",
    "wage_eur"
]

In [59]:
query_input = [
    "position",
    "playstyle",
    "min_overall",
    "max_age",
    "max_value"
]

In [60]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

feature_cols = [
    "overall", "potential", "age",
    "pace", "shooting", "passing", "dribbling",
    "defending", "physic",
    "stamina", "strength", "agility",
    "finishing", "shot_power", "positioning",
    "vision", "short_passing", "long_passing",
    "crossing", "ball_control",
    "interceptions", "standing_tackle",
    "sliding_tackle", "composure", "reactions"
]

feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols].fillna(0).astype("float32")

In [61]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [62]:
import torch

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

X_tensor.shape

torch.Size([18350, 9])

In [76]:
import torch.nn as nn
import torch.nn.functional as F

device = "mps" if torch.backends.mps.is_available() else "cpu"

X_tensor = X_tensor.to(device)

input_dim = X_tensor.shape[1]
embedding_dim = 64

In [77]:
class PlayerTower(nn.Module):
    def __init__(self, input_dim, embedding_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, embedding_dim)
        )

    def forward(self, x):
        return F.normalize(self.network(x), dim=1)

In [78]:
class QueryTower(nn.Module):
    def __init__(self, input_dim, embedding_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, embedding_dim)
        )

    def forward(self, x):
        return F.normalize(self.network(x), dim=1)

In [79]:
num_queries = 500

random_indices = torch.randint(
    0,
    X_tensor.shape[0],
    (num_queries,),
    device=device
)

query_tensor = X_tensor[random_indices].clone()

query_tensor += torch.randn_like(query_tensor) * 0.2

In [80]:
player_norm = F.normalize(X_tensor, dim=1)

similarities = query_tensor @ player_norm.T

labels = torch.zeros_like(similarities)

top_k = 10

indices = torch.topk(
    similarities,
    top_k,
    dim=1
).indices

labels.scatter_(1, indices, 1)

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], device='mps:0')

In [81]:
player_tower = PlayerTower(
    input_dim,
    embedding_dim
).to(device)

query_tower = QueryTower(
    input_dim,
    embedding_dim
).to(device)

In [82]:
optimizer = torch.optim.Adam(
    list(player_tower.parameters()) +
    list(query_tower.parameters()),
    lr=0.001
)

In [83]:
for epoch in range(1001):
    optimizer.zero_grad()

    player_emb = player_tower(X_tensor)

    query_emb = query_tower(query_tensor)

    logits = query_emb @ player_emb.T

    logits = logits * 10

    loss = F.binary_cross_entropy_with_logits(
        logits,
        labels
    )

    loss.backward()

    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 1.0347
Epoch 100, Loss: 0.0048
Epoch 200, Loss: 0.0047
Epoch 300, Loss: 0.0045
Epoch 400, Loss: 0.0045
Epoch 500, Loss: 0.0044
Epoch 600, Loss: 0.0043
Epoch 700, Loss: 0.0043
Epoch 800, Loss: 0.0042
Epoch 900, Loss: 0.0041
Epoch 1000, Loss: 0.0041


In [84]:
with torch.no_grad():
    player_embeddings = player_tower(X_tensor)

print(player_embeddings.shape)

torch.Size([18350, 64])


In [86]:
import torch

torch.save({
    "player_state": player_tower.state_dict(),
    "query_state": query_tower.state_dict(),
    "input_dim": input_dim,
    "embedding_dim": embedding_dim,
    "feature_cols": feature_cols,
    "scaler": scaler
}, "../models/two_tower.pt")

print("Saved")

Saved
